# Dirty Data Treatment Guide

This notebook is a guided data cleaning exercise using a synthetic dirty sales dataset.

The goal is not only to clean the dataset, but also to understand **why each treatment makes sense**.

The dataset contains problems such as:

- missing values;
- incorrect data types;
- inconsistent date formats;
- duplicated records;
- typos;
- invalid values;
- semantic inconsistencies between columns;
- broken characters;
- leading/trailing blank spaces;
- different units and scales.

We will treat the dataset step by step.

## 1. Import libraries and load the dataset

We read the dataset using `dtype="string"` because many columns contain dirty values.

For example, numeric columns may contain values such as `"two"`, `"$89.90"` or `"one hundred"`.
If we force numeric types too early, we may lose information about the original problem.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

df_raw = pd.read_csv("/content/retail_data.csv", dtype="string")

df = df_raw.copy()

df.head()

,Order ID,Customer ID,Customer Name,Email,Customer Since,Order Date,Delivery Date,Age,Gender,Location,Product,Category,Quantity,Unit Price,Currency,Discount (%),Total Amount,Weight,Temperature,Payment Method,Returned,Return Reason,Satisfaction Score,Sales Channel,Notes
0,ORD-1001,CUS-2011,Daniel Brown,alice@email.com,2021-07-14,01/15/2024,2024-09-08,22,Male,"Los Angeles, CA, USA",Chocolate Box,Food,2,18.5,USD,0,37.0,2400 g,18 C,Credit Card,FALSE,<NA>,5,Phone,<NA>
1,ORD-1002,CUS-2019,Daniel Brown,alice@email.com,2022-05-14,2024-02-03,2024-10-24,thirty,Female,New York - NY - USA,Running Shoes,Sports,1,120.0,USD,20,96.0,0.8 kg,<NA>,Bank Transfer,FALSE,<NA>,3,Store,<NA>
2,ORD-1003,CUS-2019,Daniel Brown,alice@email.com,2022-05-14,03-18-2024,2024-10-08,<NA>,Unknown,Denver / CO / USA,Office Chair,Furniture,1,210.0,USD,0,210.0,350 g,<NA>,Cash,FALSE,<NA>,2,Store,priority customer
3,ORD-1004,CUS-2003,Brian Smith,grace@email.com,2022-06-05,18 Apr 2024,2024-12-19,49,Prefer not to say,"Miami, FL",Milk 1L,Food,two,4.2,USD,15,7.14,13 kg,4 C,Credit Card,FALSE,<NA>,1,Mobile App,gift order
4,ORD-1005,CUS-2023,Alice Johnson,emily@email.com,2024-05-23,2024/05/21,2024-02-07,42,Non-binary,"Seattle, WA, United States",Bluetooth Speaker,Electronics,2,55.0,USD,15,93.5,1050 g,<NA>,<NA>,TRUE,Damaged,4,Mobile App,priority customer


## 2. Initial inspection

Before cleaning, we need to understand:

- number of rows and columns;
- column names;
- data types read by pandas;
- missing values;
- duplicated records;
- unique values in categorical columns.

At this point, we are not changing the dataset yet. We are diagnosing the problems.

In [2]:
df.shape

(44, 25)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Order ID            43 non-null     string
 1   Customer ID         44 non-null     string
 2   Customer Name       44 non-null     string
 3   Email               44 non-null     string
 4   Customer Since      44 non-null     string
 5   Order Date          43 non-null     string
 6   Delivery Date       44 non-null     string
 7   Age                 43 non-null     string
 8   Gender              44 non-null     string
 9   Location            41 non-null     string
 10  Product             44 non-null     string
 11  Category            43 non-null     string
 12  Quantity            44 non-null     string
 13  Unit Price          44 non-null     string
 14  Currency            43 non-null     string
 15  Discount (%)        43 non-null     string
 16  Total Amount        43 non-n

In [4]:
df.isna().sum().sort_values(ascending=False)

,0
Return Reason,38
Temperature,29
Notes,20
Location,3
Total Amount,1
Age,1
Order Date,1
Category,1
Order ID,1
Satisfaction Score,1


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df[df.duplicated(keep=False)]

,Order ID,Customer ID,Customer Name,Email,Customer Since,Order Date,Delivery Date,Age,Gender,Location,Product,Category,Quantity,Unit Price,Currency,Discount (%),Total Amount,Weight,Temperature,Payment Method,Returned,Return Reason,Satisfaction Score,Sales Channel,Notes


## Remove rows with missing Order ID

`order_id` is an identifier for each transaction.

If `order_id` is missing, we cannot reliably track, audit or remove duplicated transactions. Therefore, instead of imputing this column, we remove rows where `order_id` is missing.

In [7]:
# Order ID is mandatory because it identifies each transaction
df = df.dropna(subset=["Order ID"])

df["Order ID"].isna().sum()

np.int64(0)

## 3. Inspect unique values

This step helps us find:

- typing mistakes;
- inconsistent capitalization;
- blank spaces;
- invalid category names;
- broken characters;
- semantic problems.

For a real project, this inspection should be done before deciding how to clean categorical variables.

In [8]:
for col in df.columns:
    print("=" * 80)
    print(col)
    print(df[col].dropna().unique()[:30])

Order ID
<StringArray>
['ORD-1001', 'ORD-1002', 'ORD-1003', 'ORD-1004', 'ORD-1005', 'ORD-1006',
 'ORD-1007', 'ORD-1008', 'ORD-1009', 'ORD-1010', 'ORD-1011', 'ORD-1012',
 'ORD-1013', 'ORD-1014', 'ORD-1015', 'ORD-1016', 'ORD-1017', 'ORD-1018',
 'ORD-1019', 'ORD-1020', 'ORD-1021', 'ORD-1022', 'ORD-1024', 'ORD-1025',
 'ORD-1026', 'ORD-1027', 'ORD-1028', 'ORD-1029', 'ORD-1030', 'ORD-1031']
Length: 30, dtype: string
Customer ID
<StringArray>
['CUS-2011', 'CUS-2019', 'CUS-2003', 'CUS-2023', 'CUS-2020', 'CUS-2018',
 'CUS-2010', 'CUS-2021', 'CUS-2007', 'CUS-2013', 'CUS-2008', 'CUS-2001',
 'CUS-2022', 'CUS-2009', 'CUS-2014', 'CUS-2017', 'CUS-2024', 'CUS-2016',
 'CUS-2002', 'CUS-2026']
Length: 20, dtype: string
Customer Name
<StringArray>
[    'Daniel Brown',      'Brian Smith',    'Alice Johnson',
     'Grace Wilson',     'Henry Taylor',      'Emily Davis',
     'Carla Mendes', 'Marta GonÃ§alves']
Length: 8, dtype: string
Email
<StringArray>
['alice@email.com', 'grace@email.com', 'emily@email.co

## 4. Standardize column names

Column names with spaces, parentheses and capital letters can be harder to use in code.

We will convert them to `snake_case`.

In [9]:
def clean_column_name(col):
    col = col.strip().lower()

    cleaned = []
    previous_was_separator = False

    #easier with regex
    for char in col:
        if char.isascii() and char.isalnum():
            cleaned.append(char)
            previous_was_separator = False
        elif not previous_was_separator:
            cleaned.append("_")
            previous_was_separator = True

    return "".join(cleaned).strip("_")


df.columns = [clean_column_name(col) for col in df.columns]

df.columns

Index(['order_id', 'customer_id', 'customer_name', 'email', 'customer_since',
       'order_date', 'delivery_date', 'age', 'gender', 'location', 'product',
       'category', 'quantity', 'unit_price', 'currency', 'discount',
       'total_amount', 'weight', 'temperature', 'payment_method', 'returned',
       'return_reason', 'satisfaction_score', 'sales_channel', 'notes'],
      dtype='object')

## 5. Standardize missing value markers

Dirty datasets often represent missing or invalid values in different ways.

Examples:

- empty strings;
- `"Unknown"`;
- `"Error"`;
- `"<NA>"`;
- `"N/A"`;
- `"None"`.

Here, we replace these markers with proper pandas missing values.

Important: not every `"Unknown"` should always be removed automatically in every project.
In this teaching dataset, `"Unknown"` is being treated as missing because the column guide defines it as an invalid placeholder.

In [10]:
missing_markers = [
    "", " ", "unknown", "Unknown", "UNKNOWN",
    "error", "Error", "ERROR",
    "<NA>", "NA", "N/A", "n/a", "none", "None", "NULL", "null"
]

df = df.replace(missing_markers, pd.NA)

df.isna().sum().sort_values(ascending=False)

,0
return_reason,37
temperature,28
notes,20
location,3
total_amount,1
order_date,1
gender,1
category,1
age,1
satisfaction_score,1


## 6. Remove leading and trailing blank spaces

Blank spaces create fake categories.

For example:

- `"credit card"`
- `" credit card "`

These values look different to pandas, but they represent the same category.

In [11]:
string_cols = df.select_dtypes(include="string").columns

for col in string_cols:
    df[col] = df[col].str.strip()

# Check if spaces were removed
for col in ["customer_name", "email", "location", "product", "category", "payment_method"]:
    print(col, "->", df[col].dropna().unique())

customer_name -> <StringArray>
[    'Daniel Brown',      'Brian Smith',    'Alice Johnson',
     'Grace Wilson',     'Henry Taylor',      'Emily Davis',
     'Carla Mendes', 'Marta GonÃ§alves']
Length: 8, dtype: string
email -> <StringArray>
['alice@email.com', 'grace@email.com', 'emily@email.com', 'carla@email.com',
 'henry@email.com', 'frank@email.com']
Length: 6, dtype: string
location -> <StringArray>
[      'Los Angeles, CA, USA',        'New York - NY - USA',
          'Denver / CO / USA',                  'Miami, FL',
 'Seattle, WA, United States',              'Boston MA USA',
            'Chicago, IL, US',            'Austin, TX, USA',
     'Los Angeles , CA , USA',          'New York, CA, USA',
            'Miami - FL - US',              'Denver CO USA',
         'Seattle / WA / USA',  'Boston, MA, United States']
Length: 14, dtype: string
product -> <StringArray>
[    'Chocolate Box',     'Running Shoes',      'Office Chair',
           'Milk 1L', 'Bluetooth Speaker',      '

## 7. Fix broken characters

Some text values contain encoding problems, such as:

- `JosÃ©` instead of `José`;
- `GonÃ§alves` instead of `Gonçalves`;
- `â€™` instead of an apostrophe.

In this example, we will correct the known broken patterns.

In [12]:
encoding_corrections = {
    "JosÃ©": "José",
    "GonÃ§alves": "Gonçalves",
    "â€™": "'"
}

for col in string_cols:
    for wrong, correct in encoding_corrections.items():
        df[col] = df[col].str.replace(wrong, correct, regex=False)

df[["customer_name", "notes"]].drop_duplicates().head(20)

,customer_name,notes
0,Daniel Brown,<NA>
2,Daniel Brown,priority customer
3,Brian Smith,gift order
4,Alice Johnson,priority customer
5,Grace Wilson,call before delivery
8,Brian Smith,priority customer
9,Henry Taylor,<NA>
10,Emily Davis,priority customer
11,Emily Davis,<NA>
12,Henry Taylor,gift order


## 8. Remove duplicated records

Duplicated rows can bias analysis and machine learning models.

Here we remove exact duplicated rows.

After that, we also check duplicated `order_id`, because an order identifier should be unique according to the business rule.

In [13]:
print("Rows before removing exact duplicates:", df.shape[0])

df = df.drop_duplicates()

print("Rows after removing exact duplicates:", df.shape[0])

Rows before removing exact duplicates: 43
Rows after removing exact duplicates: 43


In [14]:
df[df.duplicated(subset=["order_id"], keep=False)].sort_values("order_id")

,order_id,customer_id,customer_name,email,customer_since,order_date,delivery_date,age,gender,location,product,category,quantity,unit_price,currency,discount,total_amount,weight,temperature,payment_method,returned,return_reason,satisfaction_score,sales_channel,notes
0,ORD-1001,CUS-2011,Daniel Brown,alice@email.com,2021-07-14,01/15/2024,2024-09-08,22,Male,"Los Angeles, CA, USA",Chocolate Box,Food,2,18.5,USD,0,37.0,2400 g,18 C,Credit Card,FALSE,<NA>,5,Phone,<NA>
42,ORD-1001,CUS-2011,Daniel Brown,alice@email.com,2021-07-14,01/15/2024,2024-09-08,22,Male,Denver CO USA,Chocolate Box,Food,2,18.5,USD,0,37.0,0.35 kg,18 C,Credit Card,FALSE,<NA>,5,Phone,<NA>
1,ORD-1002,CUS-2019,Daniel Brown,alice@email.com,2022-05-14,2024-02-03,2024-10-24,thirty,Female,New York - NY - USA,Running Shoes,Sports,1,120.0,USD,20,96.0,0.8 kg,<NA>,Bank Transfer,FALSE,<NA>,3,Store,<NA>
43,ORD-1002,CUS-2019,Daniel Brown,alice@email.com,2022-05-14,2024-02-03,2024-10-24,thirty,Female,Seattle / WA / USA,Running Shoes,Sports,1,120.0,USD,20,96.0,0.8 kg,<NA>,Bank Transfer,FALSE,<NA>,3,Store,<NA>


If duplicated `order_id` values exist, we keep the first occurrence.

This is a business decision for the exercise. In a real company, we would investigate the original source system before deleting records.

In [15]:
df = df.drop_duplicates(subset=["order_id"], keep="first")

df.shape

(41, 25)

## 9. Correct categorical values

Now we standardize categorical columns.

This includes:

- capitalization;
- typos;
- synonymous values;
- abbreviated values.

We do not use statistical imputation here. These corrections are based on business knowledge and known valid categories.

In [16]:
# Gender
gender_map = {
    "male": "Male",
    "Femail": "Female"
}

df["gender"] = df["gender"].replace(gender_map)

# Product
product_map = {
    "Cofee Maker": "Coffee Maker",
    "Bluetooth Speker": "Bluetooth Speaker"
}

df["product"] = df["product"].replace(product_map)

# Category
category_map = {
    "Electroncs": "Electronics",
    "stationery": "Stationery"
}

df["category"] = df["category"].replace(category_map)

# Payment method
payment_map = {
    "Credt Card": "Credit Card",
    "credit card": "Credit Card",
    "paypal": "PayPal"
}

df["payment_method"] = df["payment_method"].replace(payment_map)

# Sales channel
sales_channel_map = {
    "mob app": "Mobile App"
}

df["sales_channel"] = df["sales_channel"].replace(sales_channel_map)

for col in ["gender", "product", "category", "payment_method", "sales_channel"]:
    print("=" * 80)
    print(col)
    print(df[col].dropna().unique())

gender
<StringArray>
['Male', 'Female', 'Prefer not to say', 'Non-binary']
Length: 4, dtype: string
product
<StringArray>
[    'Chocolate Box',     'Running Shoes',      'Office Chair',
           'Milk 1L', 'Bluetooth Speaker',      'Coffee Maker',
          'Notebook',          'Yoga Mat']
Length: 8, dtype: string
category
<StringArray>
['Food', 'Sports', 'Furniture', 'Electronics', 'Appliances', 'Stationery']
Length: 6, dtype: string
payment_method
<StringArray>
['Credit Card', 'Bank Transfer', 'Cash', 'PayPal', 'Debit Card']
Length: 5, dtype: string
sales_channel
<StringArray>
['Phone', 'Store', 'Mobile App', 'Online']
Length: 4, dtype: string


## 10. Convert date columns

The dataset has dates in different formats.

Examples:

- `2024-02-03`;
- `03-18-2024`;
- `18 Apr 2024`;
- `2024/05/21`;
- invalid values such as `31/31/2024`;
- text values such as `before order`.

We will convert valid dates and transform invalid dates into `NaT` - not a time.

In [17]:
date_cols = ["customer_since", "order_date", "delivery_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="mixed")

df[date_cols].head(10)

,customer_since,order_date,delivery_date
0,2021-07-14,2024-01-15,2024-09-08
1,2022-05-14,2024-02-03,2024-10-24
2,2022-05-14,2024-03-18,2024-10-08
3,2022-06-05,2024-04-18,2024-12-19
4,2024-05-23,2024-05-21,2024-02-07
5,2024-02-06,NaT,2024-05-12
6,2022-04-22,2024-11-13,NaT
7,2022-07-14,2024-10-19,2024-10-23
8,2022-06-05,2024-06-20,2024-06-22
9,2023-02-05,2024-08-04,2024-08-06


In [18]:
df[date_cols].isna().sum()

,0
customer_since,0
order_date,2
delivery_date,1


## 11. Treat semantic inconsistency between dates

Business rule:

`delivery_date` cannot be earlier than `order_date`.

If delivery date is missing or invalid, but order date exists, we can impute it using a business rule.

For this exercise, we assume a standard delivery time of **3 days**.

In [19]:
invalid_delivery = df["delivery_date"] < df["order_date"]

df.loc[invalid_delivery, "delivery_date"] = pd.NaT

df["delivery_date"] = df["delivery_date"].fillna(df["order_date"] + pd.Timedelta(days=3))

df[["order_date", "delivery_date"]].head(10)

,order_date,delivery_date
0,2024-01-15,2024-09-08
1,2024-02-03,2024-10-24
2,2024-03-18,2024-10-08
3,2024-04-18,2024-12-19
4,2024-05-21,2024-05-24
5,NaT,2024-05-12
6,2024-11-13,2024-11-16
7,2024-10-19,2024-10-23
8,2024-06-20,2024-06-22
9,2024-08-04,2024-08-06


For `order_date`, we do not use mean, median or mode directly.

Because order date is central to the transaction, rows without a valid `order_date` are not reliable for this sales analysis.

Treatment decision:

- remove rows with missing `order_date`.

In [20]:
print("Rows before removing missing order_date:", df.shape[0])

df = df.dropna(subset=["order_date"])

print("Rows after removing missing order_date:", df.shape[0])

Rows before removing missing order_date: 41
Rows after removing missing order_date: 39


For `customer_since`, we can use a business rule:

`customer_since` must be before or equal to `order_date`.

If `customer_since` is missing, we impute it with the order date, meaning we assume the customer was created at the first known purchase.

In [21]:
df.loc[df["customer_since"] > df["order_date"], "customer_since"] = pd.NaT
df["customer_since"] = df["customer_since"].fillna(df["order_date"])

df[["customer_since", "order_date"]].head()

,customer_since,order_date
0,2021-07-14,2024-01-15
1,2022-05-14,2024-02-03
2,2022-05-14,2024-03-18
3,2022-06-05,2024-04-18
4,2024-05-21,2024-05-21


## 12. Convert numeric columns

Several numeric columns are stored as text because of dirty values.

Examples:

- `two`;
- `$89.90`;
- `10%`;
- `one hundred`;
- `2400 g`;
- `39 F`.

We will create specific functions for each type of numeric conversion.

Be careful to check for foreign currency, and also for different decimal parse symbols (for example, in Brazil we would use R$ and also a comma to indicate the decimal place, instead of a dot).

A tip is to generate mapping using LLMs.

In [22]:
def word_to_number(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    mapping = {
        "zero": 0,
        "one": 1,
        "two": 2,
        "three": 3,
        "four": 4,
        "five": 5,
        "one hundred": 100
    }

    if value in mapping:
        return mapping[value]

    return value


def parse_number(value):
    if pd.isna(value):
        return np.nan

    value = word_to_number(value)

    if pd.isna(value):
        return np.nan

    value = str(value)
    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace("%", "")

    return pd.to_numeric(value, errors="coerce")


df["age"] = df["age"].apply(parse_number)
df["quantity"] = df["quantity"].apply(parse_number)
df["unit_price"] = df["unit_price"].apply(parse_number)
df["total_amount"] = df["total_amount"].apply(parse_number)
df["satisfaction_score"] = df["satisfaction_score"].apply(parse_number)

df[["age", "quantity", "unit_price", "total_amount", "satisfaction_score"]].describe()

,age,quantity,unit_price,total_amount,satisfaction_score
count,37.000000,39.000000,39.000000,38.000000,38.000000
mean,41.945946,1.717949,60.805128,348.548158,3.236842
std,24.115401,1.099012,65.288126,1610.921448,1.532252
min,-5.000000,-2.000000,-10.000000,7.140000,1.000000
25%,27.000000,1.000000,7.500000,15.880000,2.000000
50%,37.000000,2.000000,35.000000,53.125000,3.000000
75%,51.000000,2.000000,89.900000,106.500000,4.000000
max,150.000000,4.000000,210.000000,9999.990000,8.000000


## 13. Convert discount to the same scale

The column `discount_%` has values in different formats:

- `10%`;
- `15%`;
- `0.15`;
- `20`.

We will convert everything to a rate between 0 and 1.

Examples:

- `10%` becomes `0.10`;
- `20` becomes `0.20`;
- `0.15` remains `0.15`.

In [23]:
def parse_discount(value):
    if pd.isna(value):
        return np.nan

    value_str = str(value).strip()

    has_percent_symbol = "%" in value_str

    value_num = parse_number(value_str)

    if pd.isna(value_num):
        return np.nan

    if has_percent_symbol:
        return value_num / 100

    if value_num > 1:
        return value_num / 100

    return value_num


df["discount_rate"] = df["discount"].apply(parse_discount)

df[["discount", "discount_rate"]].head(15)

,discount,discount_rate
0,0,0.00
1,20,0.20
2,0,0.00
3,15,0.15
4,15,0.15
6,15,0.15
7,10%,0.10
8,5,0.05
9,20,0.20
10,20,0.20


We will keep `discount_rate` and later remove the original dirty `discount_%` column.

## 14. Convert weight to kilograms

The column `weight` has different units:

- grams;
- kilograms.

We convert all values to kilograms.

In [24]:
def parse_weight_kg(value):
    if pd.isna(value):
        return np.nan

    value_str = str(value).strip().lower()

    number_chars = []
    number_started = False

    # easier with regex
    for char in value_str:
        if char.isdigit() or char in ".+-":
            number_chars.append(char)
            number_started = True
        elif number_started:
            break

    number_str = "".join(number_chars)
    number = pd.to_numeric(number_str, errors="coerce")

    if pd.isna(number):
        return np.nan

    if "kg" in value_str:
        return number

    if "g" in value_str:
        return number / 1000

    return np.nan


df["weight_kg"] = df["weight"].apply(parse_weight_kg)

df[["weight", "weight_kg"]].head(15)

,weight,weight_kg
0,2400 g,2.40
1,0.8 kg,0.80
2,350 g,0.35
3,13 kg,13.00
4,1050 g,1.05
6,1.05 kg,1.05
7,2.4 kg,2.40
8,13.0 kg,13.00
9,1.05 kg,1.05
10,0.6 kg,0.60


## 15. Convert temperature to Celsius

The column `temperature` has values in Celsius and Fahrenheit.

We convert everything to Celsius.

Formula:
C = (F-32)*5/9

This column can have null values, because only certain products require temperature to be stated, as per business rules. In a case for exploratory analysis, it would be fine. For a machine learning, we could remove this column, build two models, or even impute room temperature.

In [25]:
def parse_temperature_c(value):
    if pd.isna(value):
        return np.nan

    value_str = str(value).strip().lower()

    number_chars = []
    number_started = False

    # easier with regex
    for char in value_str:
        if char.isdigit() or char in ".+-":
            number_chars.append(char)
            number_started = True
        elif number_started:
            break

    number_str = "".join(number_chars)

    if number_str in ["", "+", "-", ".", "+.", "-."]:
        return np.nan

    number = pd.to_numeric(number_str, errors="coerce")

    if pd.isna(number):
        return np.nan

    if "f" in value_str:
        return (number - 32) * 5 / 9

    if "c" in value_str:
        return number

    return np.nan


df["temperature_c"] = df["temperature"].apply(parse_temperature_c)

df[["temperature", "temperature_c"]].head(15)

,temperature,temperature_c
0,18 C,18.0
1,<NA>,NaN
2,<NA>,NaN
3,4 C,4.0
4,<NA>,NaN
6,4 C,4.0
7,<NA>,NaN
8,<NA>,NaN
9,4 C,4.0
10,<NA>,NaN


## 16. Apply business rules to invalid numeric values

Now that numeric columns are converted, we can identify invalid values.

Business rules:

- `age` must be between 18 and 100;
- `quantity` must be greater than 0;
- `unit_price` must be greater than 0;
- `discount_rate` must be between 0 and 1;
- `satisfaction_score` must be between 1 and 5;
- `total_amount` must be greater than or equal to 0.

Invalid values are transformed into missing values before imputation.

In [26]:
df.loc[(df["age"] < 18) | (df["age"] > 100), "age"] = np.nan
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["unit_price"] <= 0, "unit_price"] = np.nan
df.loc[(df["discount_rate"] < 0) | (df["discount_rate"] > 1), "discount_rate"] = np.nan
df.loc[(df["satisfaction_score"] < 1) | (df["satisfaction_score"] > 5), "satisfaction_score"] = np.nan
df.loc[df["total_amount"] < 0, "total_amount"] = np.nan

df[["age", "quantity", "unit_price", "discount_rate", "satisfaction_score", "total_amount"]].isna().sum()

,0
age,4
quantity,2
unit_price,1
discount_rate,1
satisfaction_score,2
total_amount,1


## 17. Use product catalog as business-rule imputation

Some columns should not be imputed by mean, median or mode.

For example:

- product category depends on product;
- unit price depends on product;
- expected weight depends on product;
- expected storage temperature depends on product.

We use a product catalog (would be an external table, for example) to correct these values.

In [28]:
product_catalog = pd.read_csv("/content/product_catalog.csv")

product_catalog

,product,category,unit_price,weight_kg,temperature_c
0,Chocolate Box,Food,18.5,2.40,18.0
1,Running Shoes,Sports,120.0,0.80,18.0
2,Office Chair,Furniture,210.0,13.00,18.0
3,Milk 1L,Food,4.2,1.05,4.0
4,Bluetooth Speaker,Electronics,55.0,0.60,18.0
5,Coffee Maker,Appliances,89.9,2.40,18.0
6,Notebook,Stationery,7.5,0.20,18.0


In [29]:
# Add catalog information to the dataset
df = df.merge(
    product_catalog,
    on="product",
    how="left",
    suffixes=("", "_catalog")
)

# Impute category, unit price, weight and temperature using product catalog
df["category"] = df["category"].fillna(df["category_catalog"])
df["unit_price"] = df["unit_price"].fillna(df["unit_price_catalog"])
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg_catalog"])
df["temperature_c"] = df["temperature_c"].fillna(df["temperature_c_catalog"])

# Remove auxiliary catalog columns
df = df.drop(columns=[
    "category_catalog",
    "unit_price_catalog",
    "weight_kg_catalog",
    "temperature_c_catalog"
])

df[["product", "category", "unit_price", "weight_kg", "temperature_c"]].head(15)

,product,category,unit_price,weight_kg,temperature_c
0,Chocolate Box,Food,18.5,2.40,18.0
1,Running Shoes,Sports,120.0,0.80,18.0
2,Office Chair,Furniture,210.0,0.35,18.0
3,Milk 1L,Food,4.2,13.00,4.0
4,Bluetooth Speaker,Electronics,55.0,1.05,18.0
5,Milk 1L,Food,4.2,1.05,4.0
6,Coffee Maker,Appliances,89.9,2.40,18.0
7,Office Chair,Furniture,210.0,13.00,18.0
8,Coffee Maker,Food,4.2,1.05,4.0
9,Bluetooth Speaker,Electronics,55.0,0.60,18.0


## 18. Correct semantic inconsistency between product and category

Even when `category` is not missing, it may be wrong.

Business rule:

Each product belongs to one expected category.

Here we force the category to match the product catalog.

In [30]:
# Create expected category from the product catalog
expected_categories = product_catalog[["product", "category"]].rename(
    columns={"category": "expected_category"}
)

df = df.merge(
    expected_categories,
    on="product",
    how="left"
)

category_inconsistency = (
    df["expected_category"].notna()
    & df["category"].notna()
    & (df["category"] != df["expected_category"])
)

df.loc[category_inconsistency, "category"] = df.loc[
    category_inconsistency,
    "expected_category"
]

df = df.drop(columns=["expected_category"])

df[["product", "category"]].drop_duplicates().sort_values("product")

,product,category
4,Bluetooth Speaker,Electronics
0,Chocolate Box,Food
6,Coffee Maker,Appliances
3,Milk 1L,Food
12,Notebook,Stationery
2,Office Chair,Furniture
1,Running Shoes,Sports
15,Yoga Mat,Sports


## 19. Cleaning and splitting the location column

The `location` column contains city, state and country in a single column.

However, the format is inconsistent. Some rows use commas, others use dashes, slashes or only blank spaces.

Examples:

- `Los Angeles, CA, USA`
- `New York - NY - USA`
- `Denver / CO / USA`
- `Boston MA USA`
- `Seattle, WA, United States`

First, we standardize the separators and country names. Then, we split the column into three new columns: `city`, `state` and `country`.

After splitting, we use a business rule to correct `state`, because each city belongs to a specific state in this dataset.

In [31]:
# Standardize location text
df["location"] = df["location"].str.strip()

df["location"] = (
    df["location"]
    .str.replace(" - ", ", ", regex=False)
    .str.replace(" / ", ", ", regex=False)
    .str.replace("United States", "USA", regex=False)
    .str.replace("US", "USA", regex=False)
    .str.replace("USAA", "USA", regex=False)
)

# Fix cases where city, state and country are separated only by spaces
df["location"] = df["location"].replace({
    "Boston MA USA": "Boston, MA, USA",
    "Denver CO USA": "Denver, CO, USA"
})

# Split location into city, state and country
df[["city", "state", "country"]] = df["location"].str.split(
    ",",
    expand=True,
    n=2
)

# Remove blank spaces after splitting
df["city"] = df["city"].str.strip()
df["state"] = df["state"].str.strip()
df["country"] = df["country"].str.strip()

# Impute missing city using company's HQ
df["city"] = df["city"].fillna("Austin")

# Fix state using business rule: state depends on city
city_state_map = {
    "Los Angeles": "CA",
    "New York": "NY",
    "Denver": "CO",
    "Miami": "FL",
    "Seattle": "WA",
    "Boston": "MA",
    "Chicago": "IL",
    "Austin": "TX"
}

df["state"] = df["city"].map(city_state_map)

# Country is fixed in this dataset
df["country"] = "USA"

df[["location", "city", "state", "country"]].drop_duplicates().sort_values("city")

,location,city,state,country
6,"Austin, TX, USA",Austin,TX,USA
8,<NA>,Austin,TX,USA
13,"Boston, MA, USA",Boston,MA,USA
5,"Chicago, IL, USA",Chicago,IL,USA
2,"Denver, CO, USA",Denver,CO,USA
0,"Los Angeles, CA, USA",Los Angeles,CA,USA
7,"Los Angeles , CA , USA",Los Angeles,CA,USA
3,"Miami, FL",Miami,FL,USA
10,"Miami, FL, USA",Miami,FL,USA
1,"New York, NY, USA",New York,NY,USA


## 20. Convert returned column to boolean

The `returned` column contains different representations:

- `TRUE`;
- `FALSE`;
- `yes`;
- `No`.

We convert everything to proper boolean values.

In [32]:
returned_map = {
    "TRUE": True,
    "FALSE": False,
    "yes": True,
    "Yes": True,
    "No": False,
    "no": False
}

df["returned"] = df["returned"].map(returned_map)

df["returned"].value_counts(dropna=False)

,count
returned,
False,32
True,7


## 21. Treat semantic inconsistency between returned and return reason

Business rules:

- if `returned == False`, `return_reason` should be `"No return"`;
- if `returned == True` and `return_reason` is missing, we can use `"Unknown reason"`.

This treatment makes sense for analysis and for machine learning.

In [33]:
df.loc[df["returned"] == False, "return_reason"] = "No return"
df.loc[(df["returned"] == True) & (df["return_reason"].isna()), "return_reason"] = "Unknown reason"

df[["returned", "return_reason"]].value_counts(dropna=False)

returned  return_reason 
False     No return         32
True      Changed mind       3
          Damaged            2
          Unknown reason     2
Name: count, dtype: int64

## 22. Impute remaining values using different strategies

Now we apply different imputation strategies according to the meaning of each column. Notice that imputation might or might not be necessary for some analysis and operations, so evaluate before applying.

### Strategy summary

| Column | Strategy | Why |
|---|---|---|
| `age` | median | robust to outliers |
| `quantity` | 1 | quantity is discrete |
| `unit_price` | product catalog, then median | mostly business-rule based |
| `discount_rate` | 0 | numerical variable, may be skewed |
| `satisfaction_score` | median | ordinal numerical scale |
| `gender` | mode | categorical variable |
| `payment_method` | mode | categorical variable |
| `sales_channel` | mode | categorical variable |
| `email` | remove row or keep missing | personal identifier, not useful for ML |
| `notes` | fill with `"No notes"` | text field where missing means no note |

In [34]:
# Fixed imputation
df["discount_rate"] = df["discount_rate"].fillna(0)
df["quantity"] = df["quantity"].fillna(1)
df["currency"] = df["currency"].fillna("USD")

# Median imputation
df["age"] = df["age"].fillna(df["age"].median())
df["satisfaction_score"] = df["satisfaction_score"].fillna(df["satisfaction_score"].median())

# Mode imputation
df["gender"] = df["gender"].fillna(df["gender"].mode()[0])
df["payment_method"] = df["payment_method"].fillna(df["payment_method"].mode()[0])
df["sales_channel"] = df["sales_channel"].fillna(df["sales_channel"].mode()[0])

# Remaining product-dependent numeric columns
df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())
df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].median())
df["temperature_c"] = df["temperature_c"].fillna(df["temperature_c"].median())

# Text field
df["notes"] = df["notes"].fillna("No notes")

## 23. Recalculate total amount using business rules

`total_amount` is a derived column.

Business rule:

total amount = quantity * unit_price * (1 - discount_rate)

Because derived columns can be recalculated, this is better than imputing by mean or median.

In [35]:
# Recalculate total_amount using the business rule
df["total_amount"] = (
    df["quantity"] * df["unit_price"] * (1 - df["discount_rate"])
).round(2)

df[["quantity", "unit_price", "discount_rate", "total_amount"]].head(15)

,quantity,unit_price,discount_rate,total_amount
0,2.0,18.5,0.00,37.00
1,1.0,120.0,0.20,96.00
2,1.0,210.0,0.00,210.00
3,2.0,4.2,0.15,7.14
4,2.0,55.0,0.15,93.50
5,3.0,4.2,0.15,10.71
6,1.0,89.9,0.10,80.91
7,2.0,210.0,0.05,399.00
8,1.0,4.2,0.20,3.36
9,1.0,55.0,0.20,44.00


## 24. Drop columns that are sensitive information

Some columns are useful for auditing or identifying records, but should usually not be used as features for analysis.

Examples:

- `order_id`;
- `customer_id`;
- `customer_name`;
- `email`;
- original dirty columns already converted into clean columns.

This does not mean these columns are useless. It means they are not appropriate in this case.

In [36]:
columns_to_drop = [
    "order_id",
    "customer_id",
    "customer_name",
    "email",
    "discount",
    "weight",
    "temperature",
    "location"
]

df = df.drop(columns=columns_to_drop)

df.head()

,customer_since,order_date,delivery_date,age,gender,product,category,quantity,unit_price,currency,total_amount,payment_method,returned,return_reason,satisfaction_score,sales_channel,notes,discount_rate,weight_kg,temperature_c,city,state,country
0,2021-07-14,2024-01-15,2024-09-08,22.0,Male,Chocolate Box,Food,2.0,18.5,USD,37.00,Credit Card,False,No return,5.0,Phone,No notes,0.00,2.40,18.0,Los Angeles,CA,USA
1,2022-05-14,2024-02-03,2024-10-24,37.0,Female,Running Shoes,Sports,1.0,120.0,USD,96.00,Bank Transfer,False,No return,3.0,Store,No notes,0.20,0.80,18.0,New York,NY,USA
2,2022-05-14,2024-03-18,2024-10-08,37.0,Non-binary,Office Chair,Furniture,1.0,210.0,USD,210.00,Cash,False,No return,2.0,Store,priority customer,0.00,0.35,18.0,Denver,CO,USA
3,2022-06-05,2024-04-18,2024-12-19,49.0,Prefer not to say,Milk 1L,Food,2.0,4.2,USD,7.14,Credit Card,False,No return,1.0,Mobile App,gift order,0.15,13.00,4.0,Miami,FL,USA
4,2024-05-21,2024-05-21,2024-05-24,42.0,Non-binary,Bluetooth Speaker,Electronics,2.0,55.0,USD,93.50,PayPal,True,Damaged,4.0,Mobile App,priority customer,0.15,1.05,18.0,Seattle,WA,USA


## 25. Final type conversion

Now the dataset is clean enough to assign proper data types.

This step should usually happen after cleaning the dirty values.

In [37]:
numeric_cols = [
    "age", "quantity", "unit_price", "discount_rate",
    "total_amount", "satisfaction_score", "weight_kg", "temperature_c"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_cols = [
    "gender", "city", "state", "country", "product", "category",
    "currency", "payment_method", "return_reason", "sales_channel", "notes"
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

df["returned"] = df["returned"].astype("bool")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_since      39 non-null     datetime64[ns]
 1   order_date          39 non-null     datetime64[ns]
 2   delivery_date       39 non-null     datetime64[ns]
 3   age                 39 non-null     float64       
 4   gender              39 non-null     category      
 5   product             39 non-null     category      
 6   category            39 non-null     category      
 7   quantity            39 non-null     float64       
 8   unit_price          39 non-null     float64       
 9   currency            39 non-null     category      
 10  total_amount        39 non-null     float64       
 11  payment_method      39 non-null     category      
 12  returned            39 non-null     bool          
 13  return_reason       39 non-null     category      
 

## 26. Final validation

At the end of the cleaning process, we check:

- missing values;
- duplicated rows;
- invalid business rules;
- semantic inconsistencies;
- data types.

In [38]:
df.isna().sum().sort_values(ascending=False)

,0
customer_since,0
order_date,0
delivery_date,0
age,0
gender,0
product,0
category,0
quantity,0
unit_price,0
currency,0


In [39]:
df.duplicated().sum()

np.int64(0)

In [40]:
# Business rule checks

checks = {
    "invalid_age": ((df["age"] < 18) | (df["age"] > 100)).sum(),
    "invalid_quantity": (df["quantity"] <= 0).sum(),
    "invalid_unit_price": (df["unit_price"] <= 0).sum(),
    "invalid_discount": ((df["discount_rate"] < 0) | (df["discount_rate"] > 1)).sum(),
    "invalid_satisfaction_score": ((df["satisfaction_score"] < 1) | (df["satisfaction_score"] > 5)).sum(),
    "delivery_before_order": (df["delivery_date"] < df["order_date"]).sum()
}

checks

{'invalid_age': np.int64(0),
 'invalid_quantity': np.int64(0),
 'invalid_unit_price': np.int64(0),
 'invalid_discount': np.int64(0),
 'invalid_satisfaction_score': np.int64(0),
 'delivery_before_order': np.int64(0)}

## 27. Save the cleaned dataset

`clean_dataset_full.csv`: cleaned dataset with identifiers still available.

In [41]:
df.to_csv("clean_dataset_full.csv", index=False)

print("Files saved successfully.")

Files saved successfully.


# Final notes

This cleaning process used several types of treatment:

| Problem                  | Example treatment                                                           |
| ------------------------ | --------------------------------------------------------------------------- |
| Missing values           | median, mode, business rule, row removal, column removal                    |
| Incorrect data types     | numeric, date and boolean conversion                                        |
| Format inconsistency     | dates, percentages, currency values, location formats                       |
| Duplicates               | remove exact duplicated rows and duplicated IDs                             |
| Typos                    | categorical mapping for values such as product, category and payment method |
| Invalid values           | transform invalid values based on businesse rules into missing values before imputation              |
| Semantic inconsistency   | product/category, order/delivery date, returned/return reason, city/state   |
| Broken characters        | string replacement                                                          |
| Blank spaces             | `str.strip()`                                                               |
| Different units/scales   | grams to kg, Fahrenheit to Celsius, percent to rate                         |
| Derived values           | recalculate `total_amount` from quantity, unit price and discount           |
| Feature extraction       | split `location` into `city`, `state` and `country`                         |
| Feature selection        | remove identifiers and raw dirty columns before analysis            |
| Business-rule correction | use product catalog and city/state rules to correct values                  |
